In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
#descargar  "dataset_pequeno.csv"
#descargar  "gerencial_competencia_2026.csv.gz"


In [1]:
PARAM <- list()
PARAM$semilla_primigenia <- 327923

PARAM$experimento <- 6320
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

In [2]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [6]:
require("data.table")

# ---------------------------------------------------------------------
# 1. Definición de archivos y parámetros
# ---------------------------------------------------------------------
archivos_pred <- c(
  "../WF6301/prediccion.txt",
  "../WF6302/prediccion.txt",
  "../WF6303/prediccion.txt",
  "../WF6304/prediccion.txt",
  "../WF6305/prediccion.txt"
)

# Parámetros para Kaggle
competencia <- "utn-2026-virtual-mgr"
cortes <- seq(800, 1300, by = 50)
nombre_ensamble <- "ensamble5"

# ---------------------------------------------------------------------
# 2. Carga y promedio de probabilidades
# ---------------------------------------------------------------------
# Lee cada archivo y los concatena en una sola tabla larga
lista_tablas <- lapply(archivos_pred, fread)
tb_todas <- rbindlist(lista_tablas)

# Calcula el promedio simple de 'prob' agrupado por 'numero_de_cliente'
tb_prediccion <- tb_todas[, .(prob = mean(prob)), by = numero_de_cliente]

# Guarda el archivo consolidado del ensamble
fwrite(tb_prediccion,
  file = "prediccion_ensamble.txt",
  sep = "\t"
)


In [8]:

# ---------------------------------------------------------------------
# 3. Generación de envíos y Submit a Kaggle
# ---------------------------------------------------------------------
# Ordena de mayor a menor probabilidad estimada
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings = FALSE)

for (envios in cortes) {

  # Asigna 0 a todos y 1 a los primeros N clientes con mayor probabilidad
  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA_", nombre_ensamble, "_", envios, ".csv")

  # Guarda el archivo en formato Kaggle (numero_de_cliente, Predicted)
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  # Construcción del comando CLI de Kaggle
  comando     <- "kaggle competitions submit"
  comp_arg    <- paste("-c", competencia)
  arch_arg    <- paste("-f", archivo_kaggle)
  mensaje_arg <- paste0("-m 'ensamble 5 modelos | envios=", envios, "'")

  linea <- paste(comando, comp_arg, arch_arg, mensaje_arg)

  cat(format(Sys.time(), "%X"), "Subiendo corte:", envios, "...\n")

  # Espera de 30 segundos para respetar los límites de la API de Kaggle
  Sys.sleep(30)
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")
}

08:04:03 PM Subiendo corte: 800 ...
08:04:38 PM Subiendo corte: 850 ...
08:05:12 PM Subiendo corte: 900 ...
08:05:47 PM Subiendo corte: 950 ...
08:06:20 PM Subiendo corte: 1000 ...
08:06:53 PM Subiendo corte: 1050 ...
08:07:27 PM Subiendo corte: 1100 ...
08:08:01 PM Subiendo corte: 1150 ...
08:08:35 PM Subiendo corte: 1200 ...
08:09:09 PM Subiendo corte: 1250 ...
08:09:42 PM Subiendo corte: 1300 ...
